# w9_packageview.ipynb — 双包分形 i2ce（pk2 家族，@g1024/2000ep）

User design: 在 pk（两个 512 包）上做 i2ce，子视图上做 CE。The g1024
anchor is split into two 512 halves -- **sentence-identical to
i2ce@1024's anchor**, so pk-vs-1024 isolates the AGGREGATION alone
(embedding-mean of two I-tied small pools vs one joint attention pool).
Pack level: per-pack CE vs the normalized-mean gallery (each 512-pack must
identify its game by itself; the anchor side gets its own discriminative
objective -- an anti-gaming lever) + pack-I x2. Views: CE only (`vce`,
user's original) or full i2ce (`vi2ce`, the paired cell that keeps view-I;
512 evidence says view-I is worth +.026 tag). Eval gallery =
normalize(mean(eA, eB)) -- the arm's own deployment recipe.
Comparison rows: ce/i2ce @512 and @1024 (same budget, joint pool).
~12G/tower, ~5h/2000ep; both towers pack on one A100. ZS-only. AUTO-STOPS.


In [ ]:
# constants
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_out"          # fixed-split campaign dir

ARMS = ["wcle_pk2i2cevce_icetf",       # user original: views = CE only
        "wcle_pk2i2cevi2ce_icetf",     # paired: views keep i2ce
        "wcle_pk2i2cesgvce_icetf"]     # sg boundary: views chase DETACHED mean
CAP = 1024                             # total budget; worker splits 2 x 512
EPOCHS = 2000
os.makedirs(OUT_DIR, exist_ok=True)
print("towers:", [f"w9_{a}_g{CAP}" for a in ARMS], f"@ {EPOCHS}ep")


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (llm views not needed).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz",
            "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)


In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# Run both towers (one per GPU; they also fit together on one 80G card
# but two GPUs is the simple default). ZS-only done marker = ep{EPOCHS} npz.
import os, subprocess, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
gpus = J.detect_gpus()
todo = []
for arm in ARMS:
    nm = J.fs_label(arm, CAP, False, 0, "clean", 16)
    if (Path(OUT_DIR) / f"tower_{nm}_fp_ep{EPOCHS}.npz").exists():
        print(f"[skip] {nm} done"); continue
    todo.append((arm, nm))
print(f"{len(todo)} tower(s) to run")

stop_evt = threading.Event()
threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True).start()
fails = []

def run_one(g, arm, nm):
    if not J.try_claim(cdir, nm):
        # corpse window: a pod that died <120s ago still looks alive.
        # Wait out DEAD_SEC once and retry before giving up (fast relaunch
        # otherwise skips everything and auto-stops -- looks like a crash).
        print(f"[claim] {nm} fresh/held -- waiting 130s for the corpse "
              "window, then retrying once", flush=True)
        time.sleep(130)
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skipped", flush=True); return
    cmd = ["python", "-u", J.FS_WORKER, "--data-dir", DATA_DIR, "--out-dir",
           OUT_DIR, "--repo", REPO, "--arm", arm, "--anchor-cap", str(CAP),
           "--epochs", str(EPOCHS), "--ckpt-every", str(J.CKPT_EVERY),
           "--ckpt-seeds", str(J.FS_CKPT_SEEDS),
           "--topup-seeds", str(J.TOPUP_SEEDS),
           "--full-pool", "--full-pool-path", FULL_POOL_PATH,
           "--claim-file", str(cdir / f"{nm}.claim")]
    print(f"[gpu{g}] start {nm}", flush=True)
    t0 = time.time()
    with open(logd / f"{arm}_g{CAP}.log", "w") as fh:
        p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                           env=dict(os.environ, CUDA_VISIBLE_DEVICES=g))
    if p.returncode != 0:
        (cdir / f"{nm}.claim").unlink(missing_ok=True); fails.append(nm)
    print(f"[gpu{g}] {'ok' if p.returncode == 0 else 'FAIL'} {nm} "
          f"[{(time.time() - t0) / 3600:.1f}h]", flush=True)

ths = [threading.Thread(target=run_one, args=(gpus[i % len(gpus)], arm, nm))
       for i, (arm, nm) in enumerate(todo)]
for i, t in enumerate(ths):
    if i:
        time.sleep(120)   # stagger starts: each worker's data-load phase has
        # an ~8.5G host-RAM transient (POOL np.load) + array loads; three
        # simultaneous startups stack the peak -- suspect in the container-OOM
        # pod deaths (attempt1 died at ~ep50, right after triple load).
    t.start()
for t in ths:
    t.join()
stop_evt.set()
print(f"done; {len(fails)} failed")
for nm in fails:
    print("  FAILED:", nm)


In [ ]:
# Readout: twin-pack pair vs the joint-pool references (ZSbest-primary).
import json
import numpy as np
from pathlib import Path
VORD = ["neutral", "noname", "positive", "negative"]
ROWS = [("wcle_pk2i2cevce_icetf_g1024", "pk2i2cevce@1024 (2x512, vce)"),
        ("wcle_pk2i2cevi2ce_icetf_g1024", "pk2i2cevi2ce@1024 (+view-I)"),
        ("wcle_pk2i2cesgvce_icetf_g1024", "pk2i2cesgvce@1024 (sg boundary)"),
        ("wcle_i2ce_icetf_g1024", "i2ce@1024 (joint pool ref)"),
        ("wcle_ce_cetf_g1024", "ce@1024 (joint pool ref)"),
        ("wcle_i2ce_icetf", "i2ce@512 (healthy ref)"),
        ("wcle_ce_cetf", "ce@512 (ref)")]
def _row(lab, nm):
    zb = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
    zp = Path(OUT_DIR) / f"zs_traj_{nm}_fp.json"
    ft = Path(OUT_DIR) / f"ft4var_{nm}_fp_best.json"
    if zb.exists():
        d = json.loads(zb.read_text())
        m4z = np.mean([d["nm_" + v] for v in VORD])
        line = (f"{lab:30s} ZSbest@ep{d['best_ep']:>4}(val) "
                + " ".join(f"{v[:3]}:{d['nm_' + v]:.3f}" for v in VORD)
                + f" m4z:{m4z:.3f} tag:{d['tag_neutral']:.3f}/{d['tag_noname']:.3f}")
    elif zp.exists():
        tr = json.loads(zp.read_text())
        eps = sorted(tr, key=lambda k: int(k[2:]))
        pk = max(eps, key=lambda k: tr[k]["nm_neutral"])
        line = (f"{lab:30s} ZS test-peak*@{pk[2:]:>4} neu {tr[pk]['nm_neutral']:.3f}"
                f" non {tr[pk]['nm_noname']:.3f}"
                f" tag {tr[pk]['tag_neutral']:.3f}/{tr[pk]['tag_noname']:.3f}")
    else:
        return f"{lab:30s} (pending)"
    if ft.exists():
        d2 = json.loads(ft.read_text())
        m4 = np.mean([np.mean([x[v]["h1"] for x in d2["per_seed"]]) for v in VORD])
        line += f" | FT m4 {m4:.3f}"
    return line

for arm, lab in ROWS:
    print(_row(lab, f"w9_{arm}"))


In [ ]:
# AUTO-STOP the pod (results are on the network volume).
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- stop the pod yourself.")
